In [ ]:
!pip install -U tensorflow keras


## Setup
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
import keras
import os

from sklearn.model_selection import train_test_split

os.environ["KERAS_BACKEND"] = "tensorflow"
keras.utils.set_random_seed(42)

In [ ]:
# Not needed because we are using a public RoBERTa model
# from huggingface_hub import login
# from google.colab import userdata

# HF_TOKEN = userdata.get('HF_TOKEN') # Access the saved secret
# if HF_TOKEN:
#     login(token=HF_TOKEN)
#     print("Successfully logged in to Hugging Face!")
# else:
#     print("HF_TOKEN not set in Colab secrets.")

## Load Dataset

In [ ]:
!pip install --upgrade kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("eshummalik/socialbuzz-sentiment-analytics")

print("Path to dataset files:", path)

In [ ]:
files = os.listdir(path)
print("Files found:", files)

In [ ]:
df = pd.read_csv(os.path.join(path, "sentimentdataset.csv"))

print(df.head())

## Demoji and Tokenize

In [ ]:
!pip install demoji

import demoji

In [ ]:
df['cleaned_text'] = df['Text'].apply(lambda x: demoji.replace(x, "")) # without emoji
df['replaced_text'] = df['Text'].apply(lambda x: demoji.replace_with_desc(x)) # textualised emoji
df['Emoji'] =  df['Text'].apply(lambda x: list(demoji.findall(x).values()))

## df['Likes'].min() -> 10.0, df['Likes'].max() -> 80.0
df['Likes'] = df['Likes'].apply(lambda x: 'High' if x > 59 else 'Low')

df = df[['Likes', 'cleaned_text', 'replaced_text', 'Emoji']]

df.head()

In [ ]:
df.shape

## Load the model and tokenizer

In [ ]:
!pip install transformers==4.40.0

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from transformers import TFRobertaModel

MODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
config = AutoConfig.from_pretrained(MODEL)

backbone = TFRobertaModel.from_pretrained(MODEL)
backbone.count_params()

In [ ]:
def preprocess(text):
    new_text = []
    for t in str(text).split(" "): # Added str() just in case of NaNs
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

df['cleaned_text'] = df['cleaned_text'].apply(preprocess)
df['replaced_text'] = df['replaced_text'].apply(preprocess)

# Tokenize 'cleaned_text' and store in DF

clean_encodings = tokenizer(df['cleaned_text'].tolist(), padding=True, truncation=True, return_tensors='pt')
df['cleaned_input_ids'] = list(clean_encodings['input_ids'])
df['cleaned_attention_mask'] = list(clean_encodings['attention_mask'])

# Tokenize 'replaced_text' and store in DF
replaced_encodings = tokenizer(df['replaced_text'].tolist(), padding=True, truncation=True, return_tensors='pt')
df['replaced_input_ids'] = list(replaced_encodings['input_ids'])
df['replaced_attention_mask'] = list(replaced_encodings['attention_mask'])

labels = (df["Likes"] == "High").astype(int).values

In [ ]:
cleaned_df = df[['cleaned_text', 'cleaned_input_ids', 'cleaned_attention_mask', 'Likes']]
replaced_df = df[['replaced_text', 'replaced_input_ids', 'replaced_attention_mask', 'Likes']]

cleaned_df.head()

In [ ]:
# Check the actual lengths of tokenized rows

def get_lengths(mask_column):
    lengths = [sum(m) for m in mask_column]
    return max(lengths), sum(lengths)/len(lengths)


max_len, avg_len = get_lengths(df['cleaned_attention_mask'])
print(f"Cleaned max: {max_len}, avg: {avg_len:.2f}")

max_len, avg_len = get_lengths(df['replaced_attention_mask'])
print(f"Replaced max: {max_len}, avg: {avg_len:.2f}")

## Split Dataset

In [ ]:
# Statified by "Likes" (binary: 1,0)
labels = (df["Likes"] == "High").astype(int)

# Train set index
train_idx, temp_idx = train_test_split(
    df.index,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

# Validation set index + Test set index
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=labels[temp_idx],
    random_state=42
)


# cleaned_df dataset
c_train_df = cleaned_df.loc[train_idx]
c_val_df   = cleaned_df.loc[val_idx]
c_test_df  = cleaned_df.loc[test_idx]

# replaced_df dataset
r_train_df = replaced_df.loc[train_idx]
r_val_df   = replaced_df.loc[val_idx]
r_test_df  = replaced_df.loc[test_idx]

# labels
y_train = labels.loc[train_idx].values
y_val   = labels.loc[val_idx].values
y_test  = labels.loc[test_idx].values

In [ ]:
c_train_df.head()

In [ ]:
r_train_df.head()

## Load a Pre-Trained Model

## Finetuning

In [ ]:
# Compute weights to correct imbalanced dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print(class_weights)

In [ ]:
import keras_hub

In [ ]:
!pip install keras_tuner

In [ ]:
class RobertaLayer(keras.layers.Layer):
    def __init__(self, backbone, **kwargs):
        super().__init__(**kwargs)
        self.backbone = backbone

    def call(self, inputs):
        input_ids, attention_mask = inputs
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            training=False
        )
        return outputs.last_hidden_state[:, 0, :]

In [ ]:
import keras_tuner as kt

def build_model(hp):
    # Define hyper-parameters
    units    = hp.Choice('units', values=[32, 64, 128])
    dropout  = hp.Choice('dropout', values=[0.1, 0.2, 0.3])
    lr       = hp.Choice('learning_rate', values=[1e-3, 1e-4, 1e-5])

    backbone.trainable = True

    input_ids      = keras.layers.Input(shape=(41,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.layers.Input(shape=(41,), dtype=tf.int32, name="attention_mask")

    cls_output = RobertaLayer(backbone)([input_ids, attention_mask])

    x   = keras.layers.Dense(units, activation='relu')(cls_output)
    x   = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs=[input_ids, attention_mask], outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# GridSearch
tuner_C = kt.GridSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=None,
    directory='my_dir_C',
    project_name='keras_grid_search_C'
)

In [ ]:
### MODEL 1: Without Emoji
# Prepare data for cleaned
x_train_C = {
    "input_ids":      tf.constant([t.numpy() for t in c_train_df["cleaned_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in c_train_df["cleaned_attention_mask"]], dtype=tf.int32)
}
x_val_C = {
    "input_ids":      tf.constant([t.numpy() for t in c_val_df["cleaned_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in c_val_df["cleaned_attention_mask"]], dtype=tf.int32)
}

tuner_C.search(
    x=x_train_C,
    y=y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_val_C, y_val),
    class_weight=class_weights
)

# Check best parameters
best_hp = tuner_C.get_best_hyperparameters(1)[0]
print("Best units:", best_hp.get('units'))
print("Best dropout:", best_hp.get('dropout'))
print("Best lr:", best_hp.get('learning_rate'))

In [ ]:
# Make the best model
cleaned_best_model = tuner_C.get_best_models(num_models=1)[0]

#fit
historyC = cleaned_best_model.fit(
    x_train_C,
    y_train,
    validation_data=(x_val_C, y_val),
    epochs=5,
    batch_size=32,
    class_weight=class_weights
)

# Prepare test data
x_test_C = {
    "input_ids":      tf.constant([t.numpy() for t in c_test_df["cleaned_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in c_test_df["cleaned_attention_mask"]], dtype=tf.int32)
}

# Evaluate
loss, accuracy = cleaned_best_model.evaluate(x_test_C, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Estimate
predictions = cleaned_best_model.predict(x_test_C)
pred_labels = (predictions > 0.5).astype(int)

In [ ]:
cleaned_best_model.summary()

**Note: The parameter count in model.summary() may show 0 for the backbone layer.
This is a known limitation of Keras 3.x, which cannot track weights of external TF models.
However, the backbone weights are still included in training and will be updated normally.

In [ ]:
### MODEL 2: Without Emoji
import keras_tuner as kt
# GridSearch
tuner_R = kt.GridSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=None,
    directory='my_dir_R',
    project_name='keras_grid_search_R'
)

# Prepare data for cleaned
x_train_R = {
    "input_ids":      tf.constant([t.numpy() for t in r_train_df["replaced_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in r_train_df["replaced_attention_mask"]], dtype=tf.int32)
}
x_val_R = {
    "input_ids":      tf.constant([t.numpy() for t in r_val_df["replaced_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in r_val_df["replaced_attention_mask"]], dtype=tf.int32)
}

tuner_R.search(
    x=x_train_R,
    y=y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_val_R, y_val),
    class_weight=class_weights
)

# Check best parameters
best_hp = tuner_R.get_best_hyperparameters(1)[0]
print("Best units:", best_hp.get('units'))
print("Best dropout:", best_hp.get('dropout'))
print("Best lr:", best_hp.get('learning_rate'))

In [ ]:
# Make the best model
replaced_best_model = tuner_R.get_best_models(num_models=1)[0]

# Prepare test data
x_test_R = {
    "input_ids":      tf.constant([t.numpy() for t in r_test_df["replaced_input_ids"]], dtype=tf.int32),
    "attention_mask": tf.constant([t.numpy() for t in r_test_df["replaced_attention_mask"]], dtype=tf.int32)
}

#fit
historyR = replaced_best_model.fit(
    x_train_R,
    y_train,
    validation_data=(x_val_R, y_val),
    epochs=5,
    batch_size=32,
    class_weight=class_weights
)

# Evaluate
loss, accuracy = replaced_best_model.evaluate(x_test_R, y_test)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Estimate
predictions = replaced_best_model.predict(x_test_R)
pred_labels = (predictions > 0.5).astype(int)

In [ ]:
replaced_best_model.summary()

**Note: The parameter count in model.summary() may show 0 for the backbone layer.
This is a known limitation of Keras 3.x, which cannot track weights of external TF models.
However, the backbone weights are still included in training and will be updated normally.

Loss Curve

In [ ]:
# Function for plotting loss curve

def plot_loss(history, title):

    train_loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs = range(1, len(train_loss)+1)

    plt.figure()

    plt.plot(epochs, train_loss, "bo", label="Training loss")
    plt.plot(epochs, val_loss, "b", label="Validation loss")

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Cross Entropy Loss")

    plt.legend()
    plt.show()

In [ ]:
plot_loss(historyC, "Model A — Text Only")
plot_loss(historyR, "Model B — Text + Emoji")

## Predictions

In [ ]:
# predict
y_prob_C = cleaned_best_model.predict(x_test_C)
y_pred_C = (y_prob_C > 0.5).astype(int)

# evaluate
cleaned_best_model.evaluate(x_train_C, y_train)
cleaned_best_model.evaluate(x_test_C, y_test)

In [ ]:
# predict
y_prob_R = replaced_best_model.predict(x_test_R)
y_pred_R = (y_prob_R > 0.5).astype(int)

# evaluate
replaced_best_model.evaluate(x_train_R, y_train)
replaced_best_model.evaluate(x_test_R, y_test)

## Evaluation metrics

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report

f1_C = f1_score(y_test, y_pred_C)
roc_C = roc_auc_score(y_test, y_prob_C)

print("Cleaned Model")
print("F1-score:", f1_C)
print("ROC-AUC:", roc_C)

print("\nClassification Report")
print(classification_report(y_test, y_pred_C, target_names=["Low", "High"]))

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, classification_report

f1_R = f1_score(y_test, y_pred_R)
roc_R = roc_auc_score(y_test, y_prob_R)

print("Cleaned Model") # should be Replaced Model
print("F1-score:", f1_R)
print("ROC-AUC:", roc_R)

print("\nClassification Report")
# correct: y_pred_R
print(classification_report(y_test, y_pred_R, target_names=["Low", "High"]))

In [ ]:
results = pd.DataFrame({

    "Model": ["Text Only", "Text + Emoji"],

    "F1 Score": [f1_C, f1_R],

    "ROC-AUC": [roc_C, roc_R]

})

results

## Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_C
)

plt.title("Confusion Matrix — Text Model")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_R
)

plt.title("Confusion Matrix — Text + Emoji Model")
plt.show()

## SHAP explanation

In [ ]:
import shap

def predict_fn(texts):
    encoded = tokenizer(
        list(texts),
        max_length=41,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )
    probs = cleaned_best_model.predict({
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"]
    })
    return probs  # shape (N, 1)

explainer_C = shap.Explainer(
    predict_fn,
    shap.maskers.Text(tokenizer)  # tokenizer 넘겨야 텍스트 마스킹 제대로 돼요
)

# train 텍스트 원본으로 넘기기
shap_values_C = explainer_C(list(c_train_df["cleaned_text"])[:20])

shap.plots.text(shap_values_C[0])

In [ ]:
def predict_fn_replaced(texts):
    encoded = tokenizer(
        list(texts),
        max_length=41,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )
    probs = replaced_best_model.predict({
        "input_ids":      encoded["input_ids"],
        "attention_mask": encoded["attention_mask"]
    })
    return probs

explainer_R = shap.Explainer(
    predict_fn_replaced,
    shap.maskers.Text(tokenizer)
)

shap_values_R = explainer_R(list(r_train_df["replaced_text"])[:20])

shap.plots.text(shap_values_R[0])